# 89 — Index Geode shot-gather datasets into the LBSSP shot catalog

This notebook builds a SQLite-compatible catalog for the Geode / streamer datasets so they can be queried alongside the nodal shot gathers produced by `90_nodal_fullnode_detection_pick_and_shotgathers.ipynb`.

It does **not** try to reprocess all Geode files. Its purpose is to create catalog tables that link:

- Jochen field-note metadata rows;
- raw or stacked Geode data files (`*.dat`, SEG-2, SEG-D, SEG-Y);
- source positions, source type, operator, and shot/gather numbers;
- nominal receiver geometries;
- file paths that later notebooks can load, plot, stack, or convert.

The catalog is SQLite-first. CSV exports are generated at the end for inspection only.


## 1. Configuration

Edit the paths below for your local filesystem. The notebook assumes the metadata workbook is Jochen's field-note spreadsheet with one sheet per line/survey.

In [ ]:
from pathlib import Path
import re
import json
import sqlite3
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd

try:
    from obspy import read, UTCDateTime
except Exception as e:
    read = None
    UTCDateTime = None
    print("ObsPy not available or failed to import:", e)

PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")

# Same database used by the nodal 90_* notebook.
CATALOG_DB = PROJECT_ROOT / "lbssp_shot_catalog.sqlite"

# Update this path if the spreadsheet lives elsewhere.
METADATA_XLSX = PROJECT_ROOT / "metadata" / "jochen_field_notes_metadata_tables.xlsx"

# Add likely Geode roots here. The notebook will scan all of them.
GEODE_SEARCH_ROOTS = [
    PROJECT_ROOT / "geode",
    PROJECT_ROOT / "Geode",
    PROJECT_ROOT / "geode_data",
    PROJECT_ROOT / "streamer",
    PROJECT_ROOT / "refraction",
    PROJECT_ROOT,
]

OUTPUT_ROOT = PROJECT_ROOT / "geode_catalog_index"
CSV_EXPORT_DIR = OUTPUT_ROOT / "catalog_exports"
LOG_DIR = OUTPUT_ROOT / "logs"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CSV_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = f"GEODE_INDEX_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"
print("RUN_ID:", RUN_ID)
print("CATALOG_DB:", CATALOG_DB)
print("METADATA_XLSX exists:", METADATA_XLSX.exists(), METADATA_XLSX)
for root in GEODE_SEARCH_ROOTS:
    print("search root:", root, "exists=", root.exists())

## 2. SQLite schema

These tables are intentionally compatible with the nodal catalog. The Geode-specific notebook will populate `shot_events`, `receiver_geometry`, `shot_gather_files`, and `trace_index` with `instrument_system='geode'` or `instrument_system='streamer'`.

In [ ]:
SCHEMA_SQL = r"""
CREATE TABLE IF NOT EXISTS processing_runs (
    run_id TEXT PRIMARY KEY,
    notebook_name TEXT,
    run_time_utc TEXT,
    input_root TEXT,
    output_root TEXT,
    parameters_json TEXT,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS shot_events (
    event_id TEXT PRIMARY KEY,
    run_id TEXT,
    line TEXT,
    survey_name TEXT,
    survey_type TEXT,
    instrument_system TEXT,
    geometry_id TEXT,
    source_x_m REAL,
    source_y_m REAL,
    source_type TEXT,
    operator TEXT,
    shot_time_utc TEXT,
    detection_time_utc TEXT,
    file_no TEXT,
    shot_no TEXT,
    is_stack INTEGER,
    n_repeated_shots INTEGER,
    metadata_sheet TEXT,
    metadata_row_index INTEGER,
    metadata_match_status TEXT,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS receiver_geometry (
    geometry_id TEXT,
    run_id TEXT,
    instrument_system TEXT,
    line TEXT,
    survey_name TEXT,
    network TEXT,
    station TEXT,
    location TEXT,
    channel_family TEXT,
    component TEXT,
    receiver_x_m REAL,
    receiver_y_m REAL,
    elevation_m REAL,
    sample_rate_hz REAL,
    serial_number TEXT,
    active_start_utc TEXT,
    active_end_utc TEXT,
    source_file TEXT,
    notes TEXT,
    PRIMARY KEY (geometry_id, station, location, component)
);

CREATE TABLE IF NOT EXISTS shot_gather_files (
    gather_id TEXT PRIMARY KEY,
    event_id TEXT,
    run_id TEXT,
    instrument_system TEXT,
    line TEXT,
    survey_name TEXT,
    component TEXT,
    file_type TEXT,
    file_path TEXT,
    original_file_path TEXT,
    n_traces INTEGER,
    n_receivers INTEGER,
    sample_rate_hz REAL,
    pre_s REAL,
    post_s REAL,
    processing_level TEXT,
    is_stack INTEGER,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS trace_index (
    trace_id TEXT PRIMARY KEY,
    gather_id TEXT,
    event_id TEXT,
    run_id TEXT,
    instrument_system TEXT,
    line TEXT,
    survey_name TEXT,
    network TEXT,
    station TEXT,
    location TEXT,
    channel TEXT,
    component TEXT,
    receiver_x_m REAL,
    receiver_y_m REAL,
    source_x_m REAL,
    source_y_m REAL,
    offset_m REAL,
    starttime_utc TEXT,
    endtime_utc TEXT,
    sampling_rate_hz REAL,
    npts INTEGER,
    file_path TEXT,
    trace_index INTEGER,
    amplitude_scale REAL,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS picks (
    pick_id TEXT PRIMARY KEY,
    event_id TEXT,
    gather_id TEXT,
    run_id TEXT,
    instrument_system TEXT,
    station TEXT,
    location TEXT,
    channel TEXT,
    component TEXT,
    receiver_x_m REAL,
    source_x_m REAL,
    offset_m REAL,
    pick_time_utc TEXT,
    pick_time_relative_s REAL,
    phase TEXT,
    picker TEXT,
    pick_quality TEXT,
    snr REAL,
    uncertainty_s REAL,
    notes TEXT
);
"""

def connect_catalog(db_path=CATALOG_DB):
    db_path = Path(db_path)
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executescript(SCHEMA_SQL)
    return conn

conn = connect_catalog()
params = {
    "metadata_xlsx": str(METADATA_XLSX),
    "geode_search_roots": [str(p) for p in GEODE_SEARCH_ROOTS],
}
conn.execute(
    """
    INSERT OR REPLACE INTO processing_runs
    (run_id, notebook_name, run_time_utc, input_root, output_root, parameters_json, notes)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """,
    (
        RUN_ID,
        "89_index_geode_datasets_into_catalog.ipynb",
        datetime.now(timezone.utc).isoformat(),
        str(PROJECT_ROOT),
        str(OUTPUT_ROOT),
        json.dumps(params, indent=2),
        "Indexes Geode / streamer shot gather files and Jochen metadata into SQLite catalog.",
    ),
)
conn.commit()
print("Schema ready")

## 3. Helper functions

In [ ]:
def normalize_column_name(name):
    """Make spreadsheet column names predictable."""
    s = str(name).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def clean_dataframe_columns(df):
    df = df.copy()
    df.columns = [normalize_column_name(c) for c in df.columns]
    return df


def first_existing(row, names, default=None):
    for name in names:
        if name in row and pd.notna(row[name]) and str(row[name]).strip() != "":
            return row[name]
    return default


def as_float_or_none(x):
    if x is None or pd.isna(x):
        return None
    try:
        return float(x)
    except Exception:
        m = re.search(r"[-+]?\d+(?:\.\d+)?", str(x))
        return float(m.group(0)) if m else None


def as_int_or_none(x):
    if x is None or pd.isna(x):
        return None
    try:
        return int(float(x))
    except Exception:
        m = re.search(r"\d+", str(x))
        return int(m.group(0)) if m else None


def canonical_file_no(x):
    if x is None or pd.isna(x):
        return None
    try:
        return str(int(float(x))).zfill(4)
    except Exception:
        s = str(x).strip()
        m = re.search(r"\d+", s)
        return m.group(0).zfill(4) if m else s


def infer_line_from_sheet(sheet_name):
    s = sheet_name.upper()
    for line in ["T1A", "T1", "T2", "T3", "T4"]:
        if re.search(rf"(^|_)({line})(_|$)", s) or s.startswith(line):
            # T1A is sometimes called T2 in notes.
            return "T1A" if line == "T2" else line
    return None


def infer_survey_type(sheet_name):
    s = sheet_name.lower()
    if "streamer" in s or "masw" in s:
        return "streamer_masw"
    if "refraction" in s or "refract" in s:
        if "1m" in s or "1_m" in s:
            return "refraction_1m"
        if "2m" in s or "2_m" in s:
            return "refraction_2m"
        return "refraction"
    return "geode"


def infer_instrument_system(sheet_name):
    s = sheet_name.lower()
    if "streamer" in s or "masw" in s:
        return "streamer"
    return "geode"


def safe_utc_string(value):
    if value is None or pd.isna(value):
        return None
    try:
        # pandas Timestamp, datetime, ISO strings, etc.
        ts = pd.to_datetime(value, utc=True)
        if pd.isna(ts):
            return None
        return ts.isoformat()
    except Exception:
        try:
            if UTCDateTime is not None:
                return UTCDateTime(value).datetime.replace(tzinfo=timezone.utc).isoformat()
        except Exception:
            return str(value)
    return None


def parse_file_no_from_name(path):
    """Best-effort extraction of a Geode file number from a filename."""
    p = Path(path)
    stem = p.stem

    # Common patterns: file123.dat, 0123.dat, shot_0123.sgy, etc.
    patterns = [
        r"(?:file|fileno|file_no|shot|shotno|shot_no)[_\- ]*(\d+)",
        r"(^|[^0-9])(\d{3,5})([^0-9]|$)",
    ]
    for pat in patterns:
        m = re.search(pat, stem, flags=re.IGNORECASE)
        if m:
            nums = [g for g in m.groups() if g and g.isdigit()]
            if nums:
                return nums[0].zfill(4)
    return None


def classify_file_type(path):
    suffix = Path(path).suffix.lower().lstrip(".")
    if suffix in {"sgy", "segy"}:
        return "segy"
    if suffix in {"sg2", "seg2"}:
        return "seg2"
    if suffix in {"sgd", "segd"}:
        return "segd"
    if suffix == "dat":
        return "geode_dat"
    return suffix or "unknown"


def make_event_id(line, survey_name, file_no=None, shot_no=None, suffix=None):
    parts = ["GEODE", line or "UNK", survey_name]
    if file_no is not None:
        parts.append(f"F{canonical_file_no(file_no)}")
    if shot_no is not None:
        parts.append(f"S{canonical_file_no(shot_no)}")
    if suffix:
        parts.append(str(suffix))
    raw = "_".join(str(p) for p in parts if p is not None)
    return re.sub(r"[^A-Za-z0-9_]+", "_", raw)


def make_geometry_id(line, survey_name, file_no=None):
    raw = f"GEOM_GEODE_{line or 'UNK'}_{survey_name}"
    if file_no is not None:
        raw += f"_F{canonical_file_no(file_no)}"
    return re.sub(r"[^A-Za-z0-9_]+", "_", raw)


def nominal_receiver_positions(receiver_first_m, receiver_last_m, receiver_spacing_m, n_receivers=None):
    first = as_float_or_none(receiver_first_m)
    last = as_float_or_none(receiver_last_m)
    spacing = as_float_or_none(receiver_spacing_m)
    n = as_int_or_none(n_receivers)

    if first is None:
        return np.array([], dtype=float)

    if n is not None and spacing is not None:
        return first + np.arange(n, dtype=float) * spacing

    if last is not None and spacing is not None and spacing != 0:
        # Include last approximately.
        n_est = int(round((last - first) / spacing)) + 1
        if n_est > 0:
            return first + np.arange(n_est, dtype=float) * spacing

    if n is not None:
        return first + np.arange(n, dtype=float)

    return np.array([first], dtype=float)


def try_read_waveform_summary(path):
    """Try to read a waveform file and return basic trace metadata. Fails gracefully."""
    if read is None:
        return None, "obspy_not_available"
    path = Path(path)
    try:
        # ObsPy can often read SEG2 and SEG-Y, but not necessarily Geode DAT or SEG-D.
        st = read(str(path))
        rows = []
        for i, tr in enumerate(st):
            rows.append({
                "trace_index": i,
                "network": getattr(tr.stats, "network", None),
                "station": getattr(tr.stats, "station", None),
                "location": getattr(tr.stats, "location", None),
                "channel": getattr(tr.stats, "channel", None),
                "starttime_utc": safe_utc_string(getattr(tr.stats, "starttime", None)),
                "endtime_utc": safe_utc_string(getattr(tr.stats, "endtime", None)),
                "sampling_rate_hz": float(getattr(tr.stats, "sampling_rate", np.nan)),
                "npts": int(getattr(tr.stats, "npts", 0)),
            })
        return pd.DataFrame(rows), None
    except Exception as e:
        return None, repr(e)


## 4. Read Jochen metadata workbook

This builds a normalized metadata table from all relevant Geode/refraction/MASW/streamer sheets. Nodal-only sheets are skipped here.

In [ ]:
if not METADATA_XLSX.exists():
    raise FileNotFoundError(
        f"Metadata workbook not found: {METADATA_XLSX}\n"
        "Update METADATA_XLSX in the configuration cell."
    )

xls = pd.ExcelFile(METADATA_XLSX)
print("Sheets:")
for s in xls.sheet_names:
    print("  ", s)

rows = []
raw_sheets = {}

skip_terms = ["nodal", "node", "review", "issue", "issues", "summary"]

for sheet in xls.sheet_names:
    sheet_l = sheet.lower()
    if any(term in sheet_l for term in skip_terms):
        continue

    try:
        df = pd.read_excel(METADATA_XLSX, sheet_name=sheet)
    except Exception as e:
        print("Could not read sheet", sheet, e)
        continue

    df = clean_dataframe_columns(df).dropna(how="all")
    raw_sheets[sheet] = df

    if df.empty:
        continue

    line = infer_line_from_sheet(sheet)
    survey_type = infer_survey_type(sheet)
    instrument_system = infer_instrument_system(sheet)
    survey_name = re.sub(r"[^A-Za-z0-9_]+", "_", sheet).strip("_")

    for idx, row in df.iterrows():
        rowd = row.to_dict()
        file_no = first_existing(rowd, ["file_no", "fileno", "file", "file_number", "record", "record_no"])
        shot_no = first_existing(rowd, ["shot_no", "shot", "shot_number", "source_no"])
        source_x = first_existing(rowd, ["source_position_m", "source_x_m", "shot_location_m", "shot_position_m", "shot_x_m", "source_location_m"])
        receiver_first = first_existing(rowd, ["receiver_first_m", "first_receiver_m", "receiver_1_m", "geophone_first_m", "spread_first_m"])
        receiver_last = first_existing(rowd, ["receiver_last_m", "last_receiver_m", "receiver_n_m", "geophone_last_m", "spread_last_m"])
        receiver_spacing = first_existing(rowd, ["receiver_spacing_m", "geophone_spacing_m", "spacing_m", "dx_m"])
        n_receivers = first_existing(rowd, ["n_receivers", "number_of_receivers", "n_channels", "channels"])
        source_type = first_existing(rowd, ["source_type", "source", "type"], default="hammer")
        operator = first_existing(rowd, ["operator", "hammer_operator", "person"])
        n_shots = first_existing(rowd, ["n_shots", "n_blows", "number_of_shots", "stack", "stack_count"])
        plate_type = first_existing(rowd, ["plate_type", "plate"])
        notes = first_existing(rowd, ["notes", "comments", "comment"])
        shot_time = first_existing(rowd, ["shot_time_utc", "time_utc", "start_time_utc", "file_time_utc", "shot_time", "time", "start_time", "file_time"])

        # Keep rows that have at least a file number, shot number, or source position.
        if file_no is None and shot_no is None and source_x is None:
            continue

        rows.append({
            "metadata_sheet": sheet,
            "metadata_row_index": int(idx),
            "line": line,
            "survey_name": survey_name,
            "survey_type": survey_type,
            "instrument_system": instrument_system,
            "file_no": canonical_file_no(file_no),
            "shot_no": canonical_file_no(shot_no),
            "source_x_m": as_float_or_none(source_x),
            "source_y_m": None,
            "receiver_first_m": as_float_or_none(receiver_first),
            "receiver_last_m": as_float_or_none(receiver_last),
            "receiver_spacing_m": as_float_or_none(receiver_spacing),
            "n_receivers": as_int_or_none(n_receivers),
            "source_type": str(source_type).strip().lower() if source_type is not None else None,
            "operator": str(operator).strip() if operator is not None else None,
            "n_repeated_shots": as_int_or_none(n_shots),
            "plate_type": str(plate_type).strip() if plate_type is not None else None,
            "shot_time_utc": safe_utc_string(shot_time),
            "notes": str(notes).strip() if notes is not None else None,
            "raw_json": json.dumps({k: str(v) for k, v in rowd.items() if pd.notna(v)}, default=str),
        })

metadata_df = pd.DataFrame(rows)
print("metadata rows:", len(metadata_df))
display(metadata_df.head(20))
print(metadata_df.groupby(["line", "survey_name", "survey_type", "instrument_system"], dropna=False).size())

## 5. Discover Geode / streamer data files

In [ ]:
DATA_EXTENSIONS = {
    ".dat", ".sg2", ".seg2", ".sgy", ".segy", ".sgd", ".segd"
}

file_rows = []
seen = set()

for root in GEODE_SEARCH_ROOTS:
    root = Path(root)
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        if p.name.startswith("._"):
            continue
        if p.suffix.lower() not in DATA_EXTENSIONS:
            continue
        # Avoid indexing products from this notebook and obvious nodal outputs.
        pstr = str(p)
        if "nodal_fullnode" in pstr or "nodal_shotgathers" in pstr:
            continue
        if p in seen:
            continue
        seen.add(p)
        file_rows.append({
            "file_path": str(p),
            "file_name": p.name,
            "suffix": p.suffix.lower(),
            "file_type": classify_file_type(p),
            "file_no": parse_file_no_from_name(p),
            "size_bytes": p.stat().st_size,
            "parent": str(p.parent),
        })

files_df = pd.DataFrame(file_rows).sort_values(["file_type", "file_name"]) if file_rows else pd.DataFrame()
print("candidate data files:", len(files_df))
if not files_df.empty:
    display(files_df.head(50))
    print(files_df["file_type"].value_counts())
    print("files with parsed file_no:", files_df["file_no"].notna().sum())

## 6. Match files to metadata rows

This uses file number where possible. If there are multiple metadata rows with the same file number, the survey root/path may need manual filtering later. For now, ambiguous matches are retained and flagged.

In [ ]:
if metadata_df.empty:
    raise RuntimeError("No metadata rows were parsed from the workbook.")

# Keep only rows with a file number for automatic file matching.
md_by_file = metadata_df[metadata_df["file_no"].notna()].copy()

if files_df.empty:
    print("No data files discovered. Continuing with metadata-only catalog.")
    matched_df = metadata_df.copy()
    matched_df["file_path"] = None
    matched_df["file_type"] = None
    matched_df["match_status"] = "metadata_only_no_files_discovered"
else:
    # Simple automatic match by file_no.
    matched = files_df.merge(
        md_by_file,
        on="file_no",
        how="right",
        suffixes=("_file", ""),
    )
    matched["match_status"] = np.where(
        matched["file_path"].notna(),
        "matched_by_file_no",
        "metadata_only_no_matching_file",
    )

    # Metadata rows without file_no cannot be matched to files automatically.
    no_file_no_md = metadata_df[metadata_df["file_no"].isna()].copy()
    if not no_file_no_md.empty:
        no_file_no_md["file_path"] = None
        no_file_no_md["file_type"] = None
        no_file_no_md["match_status"] = "metadata_only_no_file_no"
        matched = pd.concat([matched, no_file_no_md], ignore_index=True, sort=False)

    matched_df = matched

print("matched/catalog candidate rows:", len(matched_df))
cols = ["metadata_sheet", "metadata_row_index", "line", "survey_name", "file_no", "shot_no", "source_x_m", "source_type", "file_type", "file_path", "match_status"]
display(matched_df[[c for c in cols if c in matched_df.columns]].head(100))
print(matched_df["match_status"].value_counts(dropna=False))

## 7. Build catalog rows

This creates compatible rows for `shot_events`, `receiver_geometry`, `shot_gather_files`, and `trace_index`. Trace rows are generated from nominal receiver geometry first; if ObsPy can read the file, basic waveform header information is added.

In [ ]:
shot_events_rows = []
receiver_geometry_rows = []
shot_gather_file_rows = []
trace_index_rows = []
read_failures = []

for i, row in matched_df.iterrows():
    line = row.get("line")
    survey_name = row.get("survey_name")
    survey_type = row.get("survey_type")
    instrument_system = row.get("instrument_system") or infer_instrument_system(str(row.get("metadata_sheet", "")))
    file_no = row.get("file_no")
    shot_no = row.get("shot_no")
    source_x_m = as_float_or_none(row.get("source_x_m"))
    source_y_m = as_float_or_none(row.get("source_y_m"))
    event_id = make_event_id(line, survey_name, file_no=file_no, shot_no=shot_no)
    geometry_id = make_geometry_id(line, survey_name, file_no=file_no)

    n_repeated = as_int_or_none(row.get("n_repeated_shots"))
    is_stack = 1 if (n_repeated is not None and n_repeated > 1) else 0

    shot_events_rows.append({
        "event_id": event_id,
        "run_id": RUN_ID,
        "line": line,
        "survey_name": survey_name,
        "survey_type": survey_type,
        "instrument_system": instrument_system,
        "geometry_id": geometry_id,
        "source_x_m": source_x_m,
        "source_y_m": source_y_m,
        "source_type": row.get("source_type"),
        "operator": row.get("operator"),
        "shot_time_utc": row.get("shot_time_utc"),
        "detection_time_utc": None,
        "file_no": file_no,
        "shot_no": shot_no,
        "is_stack": is_stack,
        "n_repeated_shots": n_repeated,
        "metadata_sheet": row.get("metadata_sheet"),
        "metadata_row_index": as_int_or_none(row.get("metadata_row_index")),
        "metadata_match_status": row.get("match_status"),
        "notes": row.get("notes"),
    })

    # Nominal receiver geometry from spreadsheet.
    rec_x = nominal_receiver_positions(
        row.get("receiver_first_m"),
        row.get("receiver_last_m"),
        row.get("receiver_spacing_m"),
        row.get("n_receivers"),
    )

    # If no geometry was inferable, create no geometry rows here; later notebooks can update this.
    for j, x in enumerate(rec_x):
        station = f"G{j+1:03d}"
        for component in ["Z"]:  # Geode/streamer stacked files are often effectively single-component vertical.
            receiver_geometry_rows.append({
                "geometry_id": geometry_id,
                "run_id": RUN_ID,
                "instrument_system": instrument_system,
                "line": line,
                "survey_name": survey_name,
                "network": line,
                "station": station,
                "location": "",
                "channel_family": "GEODE",
                "component": component,
                "receiver_x_m": float(x),
                "receiver_y_m": None,
                "elevation_m": None,
                "sample_rate_hz": None,
                "serial_number": None,
                "active_start_utc": row.get("shot_time_utc"),
                "active_end_utc": row.get("shot_time_utc"),
                "source_file": row.get("file_path"),
                "notes": "Nominal receiver geometry inferred from Jochen metadata.",
            })

    file_path = row.get("file_path")
    file_type = row.get("file_type")
    if file_path is not None and not pd.isna(file_path):
        gather_id = f"GATHER_{event_id}_{file_type.upper()}"
        wf_summary, read_error = try_read_waveform_summary(file_path)
        if read_error:
            read_failures.append({"file_path": file_path, "error": read_error})
            n_traces = len(rec_x) if len(rec_x) else None
            n_receivers = len(rec_x) if len(rec_x) else None
            sample_rate = None
        else:
            n_traces = len(wf_summary)
            n_receivers = n_traces
            sample_rate = float(wf_summary["sampling_rate_hz"].dropna().iloc[0]) if wf_summary["sampling_rate_hz"].notna().any() else None

        shot_gather_file_rows.append({
            "gather_id": gather_id,
            "event_id": event_id,
            "run_id": RUN_ID,
            "instrument_system": instrument_system,
            "line": line,
            "survey_name": survey_name,
            "component": "Z",
            "file_type": file_type,
            "file_path": file_path,
            "original_file_path": file_path,
            "n_traces": n_traces,
            "n_receivers": n_receivers,
            "sample_rate_hz": sample_rate,
            "pre_s": None,
            "post_s": None,
            "processing_level": "stacked_or_raw_geode_file",
            "is_stack": is_stack,
            "notes": f"Indexed from {file_type}; read_error={read_error}" if read_error else f"Indexed from {file_type}",
        })

        # Trace rows: prefer real waveform summary count if readable; otherwise nominal receiver positions.
        if wf_summary is not None and not wf_summary.empty:
            for _, trr in wf_summary.iterrows():
                ti = int(trr["trace_index"])
                rx = float(rec_x[ti]) if ti < len(rec_x) else None
                trace_index_rows.append({
                    "trace_id": f"TRACE_{gather_id}_{ti:04d}",
                    "gather_id": gather_id,
                    "event_id": event_id,
                    "run_id": RUN_ID,
                    "instrument_system": instrument_system,
                    "line": line,
                    "survey_name": survey_name,
                    "network": trr.get("network") or line,
                    "station": trr.get("station") or f"G{ti+1:03d}",
                    "location": trr.get("location") or "",
                    "channel": trr.get("channel") or "GPZ",
                    "component": "Z",
                    "receiver_x_m": rx,
                    "receiver_y_m": None,
                    "source_x_m": source_x_m,
                    "source_y_m": source_y_m,
                    "offset_m": (rx - source_x_m) if rx is not None and source_x_m is not None else None,
                    "starttime_utc": trr.get("starttime_utc"),
                    "endtime_utc": trr.get("endtime_utc"),
                    "sampling_rate_hz": trr.get("sampling_rate_hz"),
                    "npts": trr.get("npts"),
                    "file_path": file_path,
                    "trace_index": ti,
                    "amplitude_scale": None,
                    "notes": "Trace indexed from readable waveform header; receiver_x_m from nominal metadata by trace order.",
                })
        else:
            for ti, rx in enumerate(rec_x):
                trace_index_rows.append({
                    "trace_id": f"TRACE_{gather_id}_{ti:04d}",
                    "gather_id": gather_id,
                    "event_id": event_id,
                    "run_id": RUN_ID,
                    "instrument_system": instrument_system,
                    "line": line,
                    "survey_name": survey_name,
                    "network": line,
                    "station": f"G{ti+1:03d}",
                    "location": "",
                    "channel": "GPZ",
                    "component": "Z",
                    "receiver_x_m": float(rx),
                    "receiver_y_m": None,
                    "source_x_m": source_x_m,
                    "source_y_m": source_y_m,
                    "offset_m": (float(rx) - source_x_m) if source_x_m is not None else None,
                    "starttime_utc": row.get("shot_time_utc"),
                    "endtime_utc": None,
                    "sampling_rate_hz": None,
                    "npts": None,
                    "file_path": file_path,
                    "trace_index": ti,
                    "amplitude_scale": None,
                    "notes": "Nominal trace row; waveform file was not readable by ObsPy in this notebook.",
                })

shot_events_df = pd.DataFrame(shot_events_rows).drop_duplicates(subset=["event_id"])
receiver_geometry_df = pd.DataFrame(receiver_geometry_rows).drop_duplicates(subset=["geometry_id", "station", "location", "component"])
shot_gather_files_df = pd.DataFrame(shot_gather_file_rows).drop_duplicates(subset=["gather_id"])
trace_index_df = pd.DataFrame(trace_index_rows).drop_duplicates(subset=["trace_id"])
read_failures_df = pd.DataFrame(read_failures)

print("shot_events:", len(shot_events_df))
print("receiver_geometry:", len(receiver_geometry_df))
print("shot_gather_files:", len(shot_gather_files_df))
print("trace_index:", len(trace_index_df))
print("read_failures:", len(read_failures_df))

display(shot_events_df.head())
display(shot_gather_files_df.head())
display(trace_index_df.head())

## 8. Write catalog tables to SQLite

The notebook uses `INSERT OR REPLACE` through pandas by deleting rows for this `RUN_ID` first. This keeps reruns clean without touching older runs.

In [ ]:
conn = connect_catalog()

# Remove prior rows with same RUN_ID if rerunning this exact notebook instance.
for table in ["shot_events", "receiver_geometry", "shot_gather_files", "trace_index", "picks"]:
    conn.execute(f"DELETE FROM {table} WHERE run_id = ?", (RUN_ID,))
conn.commit()

# Append current rows.
shot_events_df.to_sql("shot_events", conn, if_exists="append", index=False)
receiver_geometry_df.to_sql("receiver_geometry", conn, if_exists="append", index=False)
shot_gather_files_df.to_sql("shot_gather_files", conn, if_exists="append", index=False)
trace_index_df.to_sql("trace_index", conn, if_exists="append", index=False)
conn.commit()

print("SQLite write complete:", CATALOG_DB)

# Quick counts for this run.
for table in ["shot_events", "receiver_geometry", "shot_gather_files", "trace_index"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table} WHERE run_id = ?", conn, params=(RUN_ID,))["n"].iloc[0]
    print(table, n)

## 9. Export CSV snapshots for inspection

In [ ]:
for name, df in {
    "geode_shot_events": shot_events_df,
    "geode_receiver_geometry": receiver_geometry_df,
    "geode_shot_gather_files": shot_gather_files_df,
    "geode_trace_index": trace_index_df,
    "geode_waveform_read_failures": read_failures_df,
    "geode_metadata_normalized": metadata_df,
    "geode_files_discovered": files_df,
}.items():
    out = CSV_EXPORT_DIR / f"{name}_{RUN_ID}.csv"
    df.to_csv(out, index=False)
    print(out)


## 10. Example queries

These are the kinds of queries later notebooks can use to compare Geode, streamer, and nodal gathers at common shot positions.

In [ ]:
conn = sqlite3.connect(CATALOG_DB)

# Example: all Geode/streamer events near a chosen source position.
SOURCE_X = 100.0
TOL_M = 0.25

query = """
SELECT event_id, line, survey_name, survey_type, instrument_system,
       source_x_m, source_type, operator, file_no, shot_no, n_repeated_shots
FROM shot_events
WHERE source_x_m BETWEEN ? AND ?
  AND instrument_system IN ('geode', 'streamer')
ORDER BY line, source_x_m, survey_name, file_no
"""
example = pd.read_sql(query, conn, params=(SOURCE_X - TOL_M, SOURCE_X + TOL_M))
display(example)

# Example: files linked to those events.
if not example.empty:
    event_ids = tuple(example["event_id"].tolist())
    placeholders = ",".join(["?"] * len(event_ids))
    files_for_events = pd.read_sql(
        f"""
        SELECT * FROM shot_gather_files
        WHERE event_id IN ({placeholders})
        ORDER BY event_id, file_type
        """,
        conn,
        params=event_ids,
    )
    display(files_for_events)


## 11. Notes / next steps

This notebook creates the Geode/streamer side of the catalog. Later notebooks can:

1. load matching nodal and Geode events by `line` and `source_x_m`;
2. load file paths from `shot_gather_files`;
3. assemble combined super-gathers using `trace_index.receiver_x_m`;
4. apply explicit amplitude scaling between systems in a separate processing level;
5. write stacked or combined gathers back into the same catalog with new `processing_level` values.

Important: if some Geode files cannot be read directly by ObsPy, they are still indexed by path and metadata. Conversion to SEG-Y can be handled later without changing the catalog design.